# 📊 EDA: Kent's Repertory of the Homoeopathic Materia Medica

This notebook performs an initial exploratory data analysis on the digitized Kent's Repertory database.
Key exploration objectives:
1. Overview of repertory sections and total rubric counts.
2. In-depth analysis of Section 1 (**MIND**), which serves as the primary dataset for Phase 1 synthetic case generation.
3. Hierarchy depth distributions and remedy grade associations.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.kent_db import (
    get_sections,
    get_mind_rubrics,
    get_rubrics,
    get_rubric_path,
    get_remedies,
    search_rubrics,
    get_connection,
)
print('Kent-AI data modules loaded successfully!')

## 1. Repertory Overview & Section Distribution

In [ ]:
sections = get_sections()
total_rubrics = sum(s['rubric_count'] for s in sections)
print(f'Total Sections: {len(sections)}')
print(f'Total Rubrics across Repertory: {total_rubrics:,}')

# Display top 10 sections by rubric count
sorted_sections = sorted(sections, key=lambda s: s['rubric_count'], reverse=True)
print('\nTop 10 Sections by Rubric Count:')
for rank, s in enumerate(sorted_sections[:10], 1):
    print(f"{rank:2d}. {s['name']:<20} : {s['rubric_count']:>6,d} rubrics (Pages {s['printed_start']}-{s['printed_end']})")

## 2. In-Depth Analysis of Section 1 (MIND)

Section 1 contains emotional, cognitive, and psychiatric symptoms. In our research pipeline, this section is used to generate synthetic clinical cases and train the Bio_ClinicalBERT HomeoNER extractor.

In [ ]:
mind_rubrics = get_mind_rubrics()
print(f'Total MIND Rubrics: {len(mind_rubrics):,}')

# Distribution of rubric depth levels
depth_counts = {}
for r in mind_rubrics:
    d = r['depth']
    depth_counts[d] = depth_counts.get(d, 0) + 1

print('\nDepth Distribution in MIND section:')
for depth in sorted(depth_counts.keys()):
    print(f'  Depth {depth} (Level {depth+1}): {depth_counts[depth]:>5,d} rubrics')

## 3. Prominent Top-Level MIND Rubrics

In [ ]:
# Top-level rubrics have parent_id IS NULL
root_rubrics = [r for r in mind_rubrics if r['parent_id'] is None]
print(f'Top-level (Root) MIND Rubrics: {len(root_rubrics):,}')
print('Sample Root Rubrics:')
for r in root_rubrics[:15]:
    print(f'  - ID {r["id"]:>5d}: {r["label"]}')

## 4. Remedy Grades & Prominence

In Kent's Repertory:
- **Grade 3 (Bold/CAPITALS)**: Symptom verified repeatedly in provings and clinical practice.
- **Grade 2 (Italics)**: Strongly indicated and confirmed by multiple provers.
- **Grade 1 (Roman/Plain)**: Occasionally noted or clinical observation.

In [ ]:
with get_connection() as conn:
    cur = conn.cursor()
    cur.execute('''
        SELECT COALESCE(grade, grade_candidate, 1) as g, COUNT(*)
        FROM rubric_remedies
        GROUP BY g
        ORDER BY g DESC
    ''')
    grade_distribution = cur.fetchall()

print('Remedy Grade Distribution (Entire Repertory):')
for grade, count in grade_distribution:
    desc = {3: 'Grade 3 (Bold/CAPITALS)', 2: 'Grade 2 (Italics)', 1: 'Grade 1 (Roman/Plain)'}.get(grade, 'Unknown')
    print(f'  {desc}: {count:,} remedies')

## 5. Sample Rubric Inspection with Hierarchy & Remedies

In [ ]:
sample_rubric_id = 4  # ABSENT-MINDED
path = get_rubric_path(sample_rubric_id)
remedies = get_remedies(sample_rubric_id)

print(f'Rubric ID: {sample_rubric_id}')
print(f'Full Path: {path}')
print(f'Total Associated Remedies: {len(remedies)}')

grade_3_rems = [r['full_name'] or r['abbreviation'] or r['normalized'] for r in remedies if r['grade'] == 3]
grade_2_rems = [r['full_name'] or r['abbreviation'] or r['normalized'] for r in remedies if r['grade'] == 2]

print(f'\nGrade 3 Remedies ({len(grade_3_rems)}): {grade_3_rems[:8]}...')
print(f'Grade 2 Remedies ({len(grade_2_rems)}): {grade_2_rems[:8]}...')

## 6. Full-Text Search Demonstration

In [ ]:
query = 'fear crowd'
search_results = search_rubrics(query, section_id=1, limit=5)
print(f"Search results for '{query}' in MIND:")
for res in search_results:
    print(f"  - ID {res['id']:>5d}: {res['path']}")